In [1]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_classic.vectorstores import FAISS

c:\Users\Admin\OneDrive\Desktop\ML_projects\AI Tender\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_files(data_path : str) -> list:
    loader = DirectoryLoader(
        path=data_path,
        glob = '*.pdf',
        loader_cls = PyPDFLoader
    )
    documents = loader.load()
    return documents

In [3]:
docs = load_files(data_path = r"C:\Users\Admin\OneDrive\Desktop\ML_projects\AI Tender\data")

In [4]:
docs[0]

Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2026-03-05T16:18:00+05:30', 'title': '', 'source': 'C:\\Users\\Admin\\OneDrive\\Desktop\\ML_projects\\AI Tender\\data\\tenderDoc22758.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content="\x01बड\n\x01बड\n \n \n\x01ववरण\n\x01ववरण\n/\n/\nBid\n \n \nDetails\n\x01बड\n\x01बड\n \n \nबंद\nबंद\n \n \nहोने\nहोने\n \n \nक\x10\nक\x10\n \n \nतार\x13ख\nतार\x13ख\n/\n/\nसमय\nसमय\n \n \n/Bid End Date/Time\n16-03-2026 16:00:00\n\x01बड\n\x01बड\n \n \nखुलने\nखुलने\n \n \nक\x10\nक\x10\n \n \nतार\x13ख\nतार\x13ख\n/\n/\nसमय\nसमय\n \n \n/Bid Opening\nDate/Time\n16-03-2026 16:30:00\n\x01बड\n\x01बड\n \n \nपेशकश\nपेशकश\n \n \nवैधता\nवैधता\n (\n (\nबंद\nबंद\n \n \nहोने\nहोने\n \n \nक\x10\nक\x10\n \n \nतार\x13ख\nतार\x13ख\n \n \nसे\nसे\n)\n)\n/Bid Offer\nValidity (From End Date)\n90 (Days)\nमं ालय\nमं ालय\n/\n/\nरा!य\nरा!य\n \n \nका\nका\n \n \nनाम\nनाम\n/Ministry/State Name\nMinistry Of Petroleum And Natural 

In [5]:
from typing import List
from langchain_classic.schema import Document

def splitting_docs(docs : list) ->list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 100,
    )
    chunks = splitter.split_documents(documents = docs)
    return chunks

In [6]:
chunks = splitting_docs(docs)

In [7]:
print(len(chunks))
chunks[0]

27


Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2026-03-05T16:18:00+05:30', 'title': '', 'source': 'C:\\Users\\Admin\\OneDrive\\Desktop\\ML_projects\\AI Tender\\data\\tenderDoc22758.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content="\x01बड\n\x01बड\n \n \n\x01ववरण\n\x01ववरण\n/\n/\nBid\n \n \nDetails\n\x01बड\n\x01बड\n \n \nबंद\nबंद\n \n \nहोने\nहोने\n \n \nक\x10\nक\x10\n \n \nतार\x13ख\nतार\x13ख\n/\n/\nसमय\nसमय\n \n \n/Bid End Date/Time\n16-03-2026 16:00:00\n\x01बड\n\x01बड\n \n \nखुलने\nखुलने\n \n \nक\x10\nक\x10\n \n \nतार\x13ख\nतार\x13ख\n/\n/\nसमय\nसमय\n \n \n/Bid Opening\nDate/Time\n16-03-2026 16:30:00\n\x01बड\n\x01बड\n \n \nपेशकश\nपेशकश\n \n \nवैधता\nवैधता\n (\n (\nबंद\nबंद\n \n \nहोने\nहोने\n \n \nक\x10\nक\x10\n \n \nतार\x13ख\nतार\x13ख\n \n \nसे\nसे\n)\n)\n/Bid Offer\nValidity (From End Date)\n90 (Days)\nमं ालय\nमं ालय\n/\n/\nरा!य\nरा!य\n \n \nका\nका\n \n \nनाम\nनाम\n/Ministry/State Name\nMinistry Of Petroleum And Natural 

In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1329.86it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
embeddings = []
for chunk in chunks:
    embeddings.append(embedding_model.embed_query(chunk.page_content))

In [10]:
len(embeddings[0])

768

In [34]:
# Saving the embeddings into existing Vector Database Pinecone
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
pc = Pinecone(api_key = "pcsk_65tEjs_Tjiwj1TFdcU8qw5aCLuDp9MBndxC7UP9fXFB4vL5K2sZtZAUQjW61AVck1rH2PT")
# index = pc.Index('pdf')
# vector_store = PineconeVectorStore(embedding=embedding_model, index = index)

In [35]:
# create index means creating a new vector database in pinecone
from pinecone import ServerlessSpec
index_name = 'pdf-bot'
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name = index_name,
        dimension = 1024,
        metric = 'cosine',
        spec = ServerlessSpec(
            cloud = 'aws',
            region = 'us-east-1'
        )
    )
print("New Vector Store created successfully")

New Vector Store created successfully


In [36]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2726.68it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
from langchain_pinecone import PineconeVectorStore

index = pc.Index(index_name)

vector_store = PineconeVectorStore(
    embedding=embedding_model,
    index = index
)

In [38]:
vector_store.add_documents(documents = chunks)

['29788714-0c2e-44d3-961e-38b379d25d94',
 'dfb1b6e2-1764-459e-8cdc-6a7e9914bac5',
 '8eb63aac-a06c-4023-a913-8b857ec368a8',
 '4bc5ae1e-cb1d-4a90-a632-7eb49b475189',
 '1af956c3-4a62-4ac3-b8d0-e9dccb2d0799',
 'aac92d36-9e1b-4f67-8a2e-0bac7d872164',
 '1ff3287b-bca6-4afe-a30a-10e81171836f',
 '35a70945-3795-466e-8a50-c7fd95909b66',
 'f1fd8c3c-60ee-4a34-a97d-8a9f01dac498',
 '20f32ddc-3ab1-4aec-9154-b5429dff8a00',
 '269ba3ad-5f8c-4bf6-b8a2-f93a85ded6f8',
 '48c53483-feee-47ab-bd5b-dcc59dda2e7f',
 '9a20d801-6a6f-4c93-838a-39fc9a8329fa',
 '1f269348-7551-49b5-99ef-c5178dd5efd4',
 '2125b6e4-ceca-4817-aedd-a163daf88976',
 '81091549-20ca-467b-8fa2-eb6db5f8ccbf',
 '0ca0f128-5a25-4884-a266-324dc86497fa',
 'cfb5ead6-e662-4e12-960d-1290f5b86bad',
 '33938257-2d0e-42a6-bafd-49f34f4ee075',
 'e0a00d56-b4de-4388-92c0-845c76811bf9',
 '1894d048-1c6e-4e3c-863b-78f294962926',
 '5899fd2d-7b60-470f-8805-b7db5d8668de',
 'd2d85823-7ff5-4754-831e-7c1f05940078',
 'df19b163-599b-4203-af05-0a7eef872647',
 'ae95ff5f-975c-

In [39]:
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import ChatHuggingFace

prompt = PromptTemplate(
    template = '''
    You are an expert Pdf Content summarizer to explain the content to all the mothers out here respectfully and 
    and explaining the content very clearly for answering questions about the content of the document {content} question : {question}.''',
    input_variables = ['content', 'question']
)

In [43]:
query = "What are the tender policy"
retrieved_doc = vector_store.similarity_search(query, k = 10)

In [47]:
content = ''
for i in retrieved_doc:
    content += (i.page_content)
print(content)

by Ministry of Micro, Small and Medium Enterprises and its subsequent Orders/Notifications issued by concerned
Ministry. If the bidder wants to avail the Purchase preference for services, the bidder must be the Service
provider of the offered Service. Relevant documentary evidence in this regard shall be uploaded along with the
bid in respect of the offered service. If L-1 is not an MSE and MSE Service Provider (s) has/have quoted price
within L-1+ 15% of margin of purchase preference /price band as defined in the relevant policy, then 100% order
quantity will be awarded to such MSE bidder 
subject to acceptance of L1 bid price. The buyers are advised to
refer to the 
OM_No.1_4_2021_PPD_dated_18.05.2023
 for compliance of Concurrent application of Public
Procurement Policy for Micro and Small Enterprises Order, 2012 and Public Procurement (Preference to Make in
India) Order, 2017. 
Benefits of MSE will be allowed only if the credentials of the service provider are validated on-The Sell